# Acciones iniciales

In [1]:
from google.colab import drive
from pathlib import Path
import shutil
import duckdb
import pandas as pd
from collections.abc import Iterable
import re
import numpy as np
import gc
import os
from functools import partial

pd.options.mode.copy_on_write = True
pd.set_option("mode.string_storage", "pyarrow")
pd.set_option("io.parquet.engine", "pyarrow")

In [3]:
# Monstar DRIVE y generar rutas:
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
ORIGEN_DRIVE = Path('/content/drive/MyDrive/0_PROYECTO_FINAL/ESTRUCTURA_MEDALLION/2_SILVER/FINALES_SILVER')
DESTINO_DRIVE = Path('/content/drive/MyDrive/0_PROYECTO_FINAL/ESTRUCTURA_MEDALLION/3_GOLD')

LOCAL_ORIGEN = Path('/content/trabajo')
shutil.copytree(ORIGEN_DRIVE, LOCAL_ORIGEN, dirs_exist_ok=True)

con = duckdb.connect()

In [5]:
# Creas vista tabla por cada archivo parquet que tenga
for ruta in sorted(LOCAL_ORIGEN.glob('*.parquet')):
    tabla = ruta.stem.lower()
    con.sql(f"""CREATE OR REPLACE VIEW "{tabla}" AS
                SELECT * FROM read_parquet('{ruta}')""")

con.sql("SHOW TABLES").show()

┌────────────────────┐
│        name        │
│      varchar       │
├────────────────────┤
│ ejemplares         │
│ poblacion_distrito │
│ prcoc1             │
│ prcocp             │
│ prcolp             │
│ prestamos          │
│ prlebi             │
│ prprsu             │
│ registros          │
└────────────────────┘



In [6]:
con.sql("DESCRIBE prestamos")

┌────────────────┬──────────────┬─────────┬─────────┬─────────┬─────────┐
│  column_name   │ column_type  │  null   │   key   │ default │  extra  │
│    varchar     │   varchar    │ varchar │ varchar │ varchar │ varchar │
├────────────────┼──────────────┼─────────┼─────────┼─────────┼─────────┤
│ id_ejemplar    │ VARCHAR      │ YES     │ NULL    │ NULL    │ NULL    │
│ id_biblioteca  │ VARCHAR      │ YES     │ NULL    │ NULL    │ NULL    │
│ id_lector      │ VARCHAR      │ YES     │ NULL    │ NULL    │ NULL    │
│ id_red         │ VARCHAR      │ YES     │ NULL    │ NULL    │ NULL    │
│ carnet_adulto  │ VARCHAR      │ YES     │ NULL    │ NULL    │ NULL    │
│ fecha_prestamo │ TIMESTAMP_NS │ YES     │ NULL    │ NULL    │ NULL    │
│ titulo         │ VARCHAR      │ YES     │ NULL    │ NULL    │ NULL    │
│ autor          │ VARCHAR      │ YES     │ NULL    │ NULL    │ NULL    │
│ isbn_norm      │ VARCHAR      │ YES     │ NULL    │ NULL    │ NULL    │
│ id_publico     │ VARCHAR      │ YES 

In [7]:
con.sql("DESCRIBE registros")

┌──────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│ column_name  │ column_type │  null   │   key   │ default │  extra  │
│   varchar    │   varchar   │ varchar │ varchar │ varchar │ varchar │
├──────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ id_registro  │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ titulo       │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ autor        │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ anio_edicion │ SMALLINT    │ YES     │ NULL    │ NULL    │ NULL    │
│ pais_edicion │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ genero       │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ tipo_publico │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ isbn         │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ n_ejemplares │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
└──────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘

In [8]:
con.sql("DESCRIBE ejemplares")

┌───────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│  column_name  │ column_type │  null   │   key   │ default │  extra  │
│    varchar    │   varchar   │ varchar │ varchar │ varchar │ varchar │
├───────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ id_ejemplar   │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ id_registro   │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ biblioteca    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ id_biblioteca │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ loc_ejemplar  │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ tejuelo       │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ info_edad     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ info_genero   │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ info_otro     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
└───────────────┴─────────────┴─────────┴─────────┴─────────┴───

# Creación tablas agregadas

## Prestamos

In [21]:
# Tabla prestamos filtrada por periodo y sin el club de lectura.
con.sql("""
    CREATE OR REPLACE VIEW prestamos_periodo AS
    SELECT *
    FROM prestamos
    WHERE fecha_prestamo < DATE '2026-01-01'
      AND id_biblioteca <> 'bcdicl'
""")

In [34]:
# fila por mes, biblioteca y carné. Es la tabla base de los distritos y del carné.

con.sql("""
    CREATE OR REPLACE TABLE prestamos_mes_ag AS
    SELECT
        date_trunc('month', fecha_prestamo)::DATE AS mes,
        id_biblioteca,
        carnet_adulto,
        CAST(COUNT(*) AS BIGINT) AS n_prestamos
    FROM prestamos_periodo
    GROUP BY ALL
""")

In [55]:
# Préstamos de la ciudad por mes, sobre el calendario completo (96 meses).
con.sql("""
    CREATE OR REPLACE VIEW prestamos_mes_total AS
    WITH mes_ciudad AS (
        SELECT
            mes AS anio_mes,              -- en prestamos_mes_ag la columna se llama 'mes'
            SUM(n_prestamos) AS n
        FROM prestamos_mes_ag
        GROUP BY mes                      -- agrupar por la columna real, no por el alias del calendario
    )
    SELECT
        c.anio_mes,
        c.anio,
        CAST(
            CASE
                WHEN CAST(c.sin_datos    AS INTEGER) = 1 THEN NULL          -- mes sin fichero: sin dato
                WHEN CAST(c.sin_servicio AS INTEGER) = 1 THEN COALESCE(m.n, 0) -- cierre: cero real
                ELSE m.n
            END
        AS BIGINT) AS n_prestamos,
        c.sin_datos,
        c.sin_servicio
    FROM dim_calendario c
    LEFT JOIN mes_ciudad m USING (anio_mes)
    ORDER BY c.anio_mes
""")

In [56]:
# Media y mediana periodo meses reales n-94
con.sql("""
SELECT
    COUNT(n_prestamos) AS meses_validos,
    AVG(n_prestamos) AS media_mensual,
    MEDIAN(n_prestamos) AS mediana_mensual
FROM prestamos_mes_total;
""")

┌───────────────┬───────────────────┬─────────────────┐
│ meses_validos │   media_mensual   │ mediana_mensual │
│     int64     │      double       │     double      │
├───────────────┼───────────────────┼─────────────────┤
│            94 │ 173796.9255319149 │        184448.0 │
└───────────────┴───────────────────┴─────────────────┘

In [57]:
con.sql("""
    SELECT
        year(fecha_prestamo) AS anio,
        COUNT(*) AS total_prestamos,
        COUNT(DISTINCT id_ejemplar) AS ejemplares_anio
    FROM prestamos_periodo
    GROUP BY anio
    ORDER BY anio ASC
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌───────┬─────────────────┬─────────────────┐
│ anio  │ total_prestamos │ ejemplares_anio │
│ int64 │      int64      │      int64      │
├───────┼─────────────────┼─────────────────┤
│  2018 │         2040086 │          882889 │
│  2019 │         2215420 │          891401 │
│  2020 │         1061718 │          587353 │
│  2021 │         1459885 │          704403 │
│  2022 │         2142293 │          864428 │
│  2023 │         2393739 │          924075 │
│  2024 │         2568450 │          996192 │
│  2025 │         2455320 │          981732 │
└───────┴─────────────────┴─────────────────┘

## Calendario, con todos los meses incluidos los faltantes

In [28]:
# Creacion tabla calendario con todos los meses del periodo

con.sql("""
    CREATE OR REPLACE TABLE dim_calendario AS
    WITH meses AS (
        SELECT CAST(m AS DATE) AS mes
        FROM generate_series(TIMESTAMP '2018-01-01',
                            TIMESTAMP '2025-12-01',
                            INTERVAL 1 MONTH) AS t(m)
    )
    SELECT
        mes AS anio_mes,
        year(mes) AS anio,
        month(mes) AS mes_numero,
        ['enero','febrero','marzo','abril','mayo','junio','julio','agosto',
        'septiembre','octubre','noviembre','diciembre'][month(mes)] AS nombre_mes,
        CASE
          WHEN mes IN (DATE '2020-04-01', DATE '2020-05-01')
          THEN 1 ELSE 0 END AS sin_servicio,
        CASE
          WHEN mes IN (DATE '2018-01-01', DATE '2025-05-01')
          THEN 1 ELSE 0 END AS sin_datos,
    FROM meses
    ORDER BY mes
""")

In [31]:
con.sql("""
    SELECT
      count(*),
      count(DISTINCT anio_mes),
      min(anio_mes),
      max(anio_mes),
      sum(sin_servicio),
      sum(sin_datos)
    FROM dim_calendario;
""")

┌──────────────┬──────────────────────────┬───────────────┬───────────────┬───────────────────┬────────────────┐
│ count_star() │ count(DISTINCT anio_mes) │ min(anio_mes) │ max(anio_mes) │ sum(sin_servicio) │ sum(sin_datos) │
│    int64     │          int64           │     date      │     date      │      int128       │     int128     │
├──────────────┼──────────────────────────┼───────────────┼───────────────┼───────────────────┼────────────────┤
│           96 │                       96 │ 2018-01-01    │ 2025-12-01    │                 2 │              2 │
└──────────────┴──────────────────────────┴───────────────┴───────────────┴───────────────────┴────────────────┘

## Tabla bibliotecas

In [33]:
con.sql("""
    CREATE OR REPLACE TABLE dim_biblioteca AS
      SELECT
        p.id_biblioteca AS id_biblioteca,
        b.nombre_biblioteca AS nombre_biblioteca,
        b.id_distrito AS id_distrito,
        b.distrito AS distrito,
        MIN(date_trunc('month', p.fecha_prestamo)) AS primer_mes_prestamos,
        MAX(date_trunc('month', p.fecha_prestamo)) AS ultimo_mes_prestamos,
        CAST(COUNT(DISTINCT date_trunc('month', p.fecha_prestamo)) AS SMALLINT) AS meses_con_prestamos,
        CASE
          WHEN b.distrito IS NULL OR b.distrito = 'sin_referencia' THEN 1
          ELSE 0
        END AS sin_distrito,
      FROM prestamos p
      LEFT JOIN prprsu b ON b.id_biblioteca = p.id_biblioteca
      WHERE p.fecha_prestamo < '2026-01-01'
      GROUP BY p.id_biblioteca,
      b.nombre_biblioteca,
      b.id_distrito,
      b.distrito
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

## Tabla distrito-poblacion

In [43]:
con.sql("DESCRIBE poblacion_distrito")

┌─────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│   column_name   │ column_type │  null   │   key   │ default │  extra  │
│     varchar     │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ anio            │ SMALLINT    │ YES     │ NULL    │ NULL    │ NULL    │
│ id_distrito     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ distrito        │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ total_poblacion │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
└─────────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘

In [47]:
con.sql("""
    CREATE OR REPLACE TABLE dim_distrito_anio AS
      SELECT
        anio,
        id_distrito,
        distrito,
        CAST(SUM(total_poblacion) AS BIGINT) AS total_poblacion
      FROM poblacion_distrito
      WHERE anio != 2026
      GROUP BY anio, id_distrito, distrito
""")

In [48]:
con.sql("""
      SELECT *
      FROM dim_distrito_anio
""")

┌───────┬─────────────┬─────────────────────┬─────────────────┐
│ anio  │ id_distrito │      distrito       │ total_poblacion │
│ int16 │   varchar   │       varchar       │      int64      │
├───────┼─────────────┼─────────────────────┼─────────────────┤
│  2025 │ 1           │ centro              │          146349 │
│  2025 │ 3           │ retiro              │          120447 │
│  2025 │ 4           │ salamanca           │          150796 │
│  2025 │ 6           │ tetuan              │          169479 │
│  2025 │ 8           │ fuencarral-el pardo │          257495 │
│  2025 │ 9           │ moncloa-aravaca     │          126978 │
│  2025 │ 20          │ san blas-canillejas │          169990 │
│  2024 │ 1           │ centro              │          145411 │
│  2024 │ 2           │ arganzuela          │          156559 │
│  2024 │ 3           │ retiro              │          119757 │
│    ·  │ ·           │   ·                 │             ·   │
│    ·  │ ·           │   ·             

In [49]:
# La población de la ciudad es la suma de los 21 distritos de cada año.
con.sql("""
    SELECT anio,
           COUNT(*)AS n_distritos,
           COUNT(*) FILTER (WHERE total_poblacion IS NULL) AS nulos,
           SUM(total_poblacion) AS poblacion_ciudad
    FROM dim_distrito_anio
    GROUP BY anio
    ORDER BY anio
""").show()

┌───────┬─────────────┬───────┬──────────────────┐
│ anio  │ n_distritos │ nulos │ poblacion_ciudad │
│ int16 │    int64    │ int64 │      int128      │
├───────┼─────────────┼───────┼──────────────────┤
│  2018 │          21 │     0 │          3221824 │
│  2019 │          21 │     0 │          3266126 │
│  2020 │          21 │     0 │          3334730 │
│  2021 │          21 │     0 │          3312310 │
│  2022 │          21 │     0 │          3286662 │
│  2023 │          21 │     0 │          3339931 │
│  2024 │          21 │     0 │          3460491 │
│  2025 │          21 │     0 │          3527924 │
└───────┴─────────────┴───────┴──────────────────┘



# H1

In [58]:

con.sql("""
    CREATE OR REPLACE TABLE h1_indice_anual AS
    WITH anual AS (
        SELECT anio,
               CAST(SUM(n_prestamos) AS BIGINT) AS prestamos_anio,
               COUNT(n_prestamos) AS meses_con_datos,
               SUM(n_prestamos)::DOUBLE / COUNT(n_prestamos) AS promedio_mensual
        FROM prestamos_mes_total
        GROUP BY anio
    ),
    pob AS (
        SELECT anio, CAST(SUM(total_poblacion) AS BIGINT) AS poblacion
        FROM dim_distrito_anio
        GROUP BY anio
    )
    SELECT
        a.anio,
        a.prestamos_anio,
        a.meses_con_datos,
        a.promedio_mensual,
        100 * a.promedio_mensual
            / (SELECT promedio_mensual FROM anual WHERE anio = 2018) AS indice_2018,
        100 * (a.promedio_mensual
               / LAG(a.promedio_mensual) OVER (ORDER BY a.anio) - 1) AS variacion_interanual_pct,
        p.poblacion,
        1000 * a.promedio_mensual / p.poblacion  AS prestamos_mes_1000_hab,
        12   * a.promedio_mensual / p.poblacion  AS prestamos_hab_anio
    FROM anual a
    JOIN pob p USING (anio)
    ORDER BY a.anio
""")

In [59]:
# Media y mediana promedio_mensual anualizado
con.sql("""WITH anualizado AS (
    SELECT
        anio,
        promedio_mensual * 12 AS prestamos_anualizados
    FROM h1_indice_anual
    WHERE anio BETWEEN 2018 AND 2025
)
SELECT
    COUNT(*)  AS n_anios,
    ROUND(AVG(prestamos_anualizados), 1) AS media_anualizada,
    ROUND(MEDIAN(prestamos_anualizados), 1) AS mediana_anualizada
FROM anualizado;""")

┌─────────┬──────────────────┬────────────────────┐
│ n_anios │ media_anualizada │ mediana_anualizada │
│  int64  │      double      │       double       │
├─────────┼──────────────────┼────────────────────┤
│       8 │        2093198.0 │          2220484.2 │
└─────────┴──────────────────┴────────────────────┘

In [60]:
con.sql("""
    SELECT
      anio,
      prestamos_anio,
      meses_con_datos,
      ROUND(promedio_mensual, 1)  AS promedio_mensual,
      ROUND(indice_2018, 1) AS indice_2018,
      ROUND(variacion_interanual_pct, 1)  AS var_pct,
      poblacion,
      ROUND(prestamos_mes_1000_hab, 2) AS mes_1000_hab,
      ROUND(prestamos_hab_anio, 3) AS hab_anio
    FROM h1_indice_anual
    ORDER BY anio
""").show()


con.sql("""
    SELECT COUNT(*) AS n_filas,
           MAX(indice_2018) FILTER (WHERE anio = 2018) AS indice_base,
           SUM(prestamos_anio) AS total
    FROM h1_indice_anual
""").show()
con.sql("DESCRIBE h1_indice_anual").show()

┌───────┬────────────────┬─────────────────┬──────────────────┬─────────────┬─────────┬───────────┬──────────────┬──────────┐
│ anio  │ prestamos_anio │ meses_con_datos │ promedio_mensual │ indice_2018 │ var_pct │ poblacion │ mes_1000_hab │ hab_anio │
│ int64 │     int64      │      int64      │      double      │   double    │ double  │   int64   │    double    │  double  │
├───────┼────────────────┼─────────────────┼──────────────────┼─────────────┼─────────┼───────────┼──────────────┼──────────┤
│  2018 │        2040086 │              11 │         185462.4 │       100.0 │    NULL │   3221824 │        57.56 │    0.691 │
│  2019 │        2215420 │              12 │         184618.3 │        99.5 │    -0.5 │   3266126 │        56.53 │    0.678 │
│  2020 │        1061718 │              12 │          88476.5 │        47.7 │   -52.1 │   3334730 │        26.53 │    0.318 │
│  2021 │        1459885 │              12 │         121657.1 │        65.6 │    37.5 │   3312310 │        36.73 │    

## Estacionalidad

In [63]:
# ratio = préstamos del mes / promedio mensual de ese año.
con.sql("""
    CREATE OR REPLACE TABLE estacionalidad_ratios AS
    SELECT
        t.anio_mes,
        t.anio,
        c.mes_numero,
        c.nombre_mes,
        t.n_prestamos,
        AVG(t.n_prestamos) OVER (PARTITION BY t.anio)  AS promedio_mensual_anio,
        t.n_prestamos / AVG(t.n_prestamos) OVER (PARTITION BY t.anio) AS ratio
    FROM prestamos_mes_total t
    JOIN dim_calendario c USING (anio_mes)
    WHERE t.n_prestamos IS NOT NULL
""")

In [65]:
# Indice estacionalidad
con.sql("""
    CREATE OR REPLACE TABLE estacionalidad_indice AS
    SELECT
        mes_numero,
        ANY_VALUE(nombre_mes)   AS nombre_mes,
        COUNT(*)                AS n_ratios,
        median(ratio)           AS indice,
        MIN(ratio)              AS ratio_min,
        arg_min(anio, ratio)    AS anio_min,
        MAX(ratio)              AS ratio_max,
        arg_max(anio, ratio)    AS anio_max
    FROM estacionalidad_ratios
    GROUP BY mes_numero
    ORDER BY mes_numero
""")

In [66]:
# Indice estacinalidad normalizado
con.sql("""
    SELECT
        nombre_mes,
        indice AS mediana_mes,
        ROUND(AVG(indice) OVER (), 4) AS media_indices,
        ROUND(indice / AVG(indice) OVER (), 3) AS indice_normalizado
    FROM estacionalidad_indice
""").show()

┌────────────┬────────────────────┬───────────────┬────────────────────┐
│ nombre_mes │    mediana_mes     │ media_indices │ indice_normalizado │
│  varchar   │       double       │    double     │       double       │
├────────────┼────────────────────┼───────────────┼────────────────────┤
│ enero      │ 1.0010442095427203 │        1.0041 │              0.997 │
│ febrero    │ 1.0077714341362616 │        1.0041 │              1.004 │
│ marzo      │ 1.1122218200627385 │        1.0041 │              1.108 │
│ abril      │ 1.0915029613236165 │        1.0041 │              1.087 │
│ mayo       │ 0.9706102902528763 │        1.0041 │              0.967 │
│ junio      │ 0.9470897112658241 │        1.0041 │              0.943 │
│ julio      │ 0.9371787295694718 │        1.0041 │              0.933 │
│ agosto     │  0.821251937332123 │        1.0041 │              0.818 │
│ septiembre │ 1.0299264622384714 │        1.0041 │              1.026 │
│ octubre    │  1.108633016640396 │        1.0041 │

In [67]:
# CV estacionalidad
con.sql("""
    WITH normalizados AS (
        SELECT
            nombre_mes,
            indice / AVG(indice) OVER () AS idx
        FROM estacionalidad_indice
    )
    SELECT
        ROUND(STDDEV_POP(idx), 4) AS intensidad_estacional_cv,
        ROUND(MAX(idx) - MIN(idx), 2) AS amplitud_max_min,
        ROUND(MAX(idx) / MIN(idx), 2) AS ratio_pico_valle
    FROM normalizados
""").show()


┌──────────────────────────┬──────────────────┬──────────────────┐
│ intensidad_estacional_cv │ amplitud_max_min │ ratio_pico_valle │
│          double          │      double      │      double      │
├──────────────────────────┼──────────────────┼──────────────────┤
│                   0.0789 │             0.29 │             1.35 │
└──────────────────────────┴──────────────────┴──────────────────┘



## Indice carné adulto / infantil

In [68]:
con.sql("""
    CREATE OR REPLACE TABLE carnet_indice_anual AS
    WITH por_carne AS (
        SELECT
            year(mes) AS anio,
            SUM(n_prestamos) FILTER (WHERE carnet_adulto = '1') AS prest_adulto,
            SUM(n_prestamos) FILTER (WHERE carnet_adulto = '0') AS prest_infantil
        FROM prestamos_mes_ag
        GROUP BY ALL
    ),
    base AS (
        SELECT
            p.anio,
            CAST(p.prest_adulto   AS BIGINT)           AS prestamos_adulto,
            CAST(p.prest_infantil AS BIGINT)           AS prestamos_infantil,
            h.meses_con_datos,
            p.prest_adulto::DOUBLE   / h.meses_con_datos AS promedio_adulto,
            p.prest_infantil::DOUBLE / h.meses_con_datos AS promedio_infantil
        FROM por_carne p
        JOIN h1_indice_anual h USING (anio)
    )
    SELECT
        *,
        100 * promedio_adulto   / (SELECT promedio_adulto   FROM base WHERE anio = 2018) AS indice_adulto_2018,
        100 * promedio_infantil / (SELECT promedio_infantil FROM base WHERE anio = 2018) AS indice_infantil_2018,
        promedio_infantil / promedio_adulto  AS ratio_infantil_adulto
    FROM base
    ORDER BY anio
""")

# Préstamos por Obra

In [81]:
RUTA_SILVER = Path('/content/drive/MyDrive/0_PROYECTO_FINAL/ESTRUCTURA_MEDALLION/2_SILVER')

In [70]:
# Correcciones manuales de la clave de obra (Invisible). all_varchar: todo como texto, sin que DuckDB adivine tipos.
con.sql(f"""
    CREATE OR REPLACE TABLE correcciones_obra AS
    SELECT * FROM read_csv('{RUTA_SILVER}/verificacion/correcciones_obra.csv',
                           delim = ';', header = true, all_varchar = true)
""")

# Excepciones de tipo de obra (vacía, pero se carga para que la cadena quede completa y reutilizable).
# columns = ...: con solo la cabecera, así el esquema no depende de lo que adivine DuckDB.
con.sql(f"""
    CREATE OR REPLACE TABLE excepciones_tipo_obra AS
    SELECT * FROM read_csv('{RUTA_SILVER}/verificacion/excepciones_tipo_obra.csv',
                           delim = ';', header = true,
                           columns = {{'titulo': 'VARCHAR', 'autor': 'VARCHAR', 'tipo_corregido': 'VARCHAR',
                                       'anio_publicacion_real': 'VARCHAR', 'fuente': 'VARCHAR', 'nota': 'VARCHAR'}})
""")

# Revisión de las 133 obras del top anterior: tipo_fusion (lista de exclusión) y titulo_mostrar.
# COALESCE(titulo, ''): los títulos vacíos (1984, 4 3 2 1) llegan como NULL al leer el CSV y en préstamos son ''.
con.sql(f"""
    CREATE OR REPLACE TABLE revision_top_anterior AS
    SELECT * REPLACE (COALESCE(titulo, '') AS titulo)
    FROM read_csv('{RUTA_SILVER}/extras_limpieza/REVISION_TOP/revisadas/revision_fusiones_top_revisada.csv',
                  delim = ',', header = true, quote = '"', all_varchar = true)
""")

# Controles: 1 corrección, 0 excepciones, 133 obras revisadas con 25 series y 4 guías
con.sql("SELECT COUNT(*) AS n_correcciones FROM correcciones_obra").show()
con.sql("SELECT COUNT(*) AS n_excepciones FROM excepciones_tipo_obra").show()
con.sql("""
    SELECT COALESCE(tipo_fusion, 'ninguna') AS tipo_fusion, COUNT(*) AS n
    FROM revision_top_anterior
    GROUP BY ALL ORDER BY ALL
""").show()

┌────────────────┐
│ n_correcciones │
│     int64      │
├────────────────┤
│              1 │
└────────────────┘

┌───────────────┐
│ n_excepciones │
│     int64     │
├───────────────┤
│             0 │
└───────────────┘

┌─────────────┬───────┐
│ tipo_fusion │   n   │
│   varchar   │ int64 │
├─────────────┼───────┤
│ guía        │     4 │
│ ninguna     │   104 │
│ serie       │    25 │
└─────────────┴───────┘



## Préstamos por obra

In [72]:
con.sql("""
    CREATE OR REPLACE VIEW prestamos_obra AS
    SELECT
        p.* REPLACE (
            COALESCE(c.titulo_destino, p.titulo) AS titulo,
            COALESCE(c.autor_destino,  p.autor)  AS autor
        ),
        md5(COALESCE(c.titulo_destino, p.titulo) || ' | ' || COALESCE(c.autor_destino, p.autor)) AS id_obra
    FROM prestamos_periodo p
    LEFT JOIN correcciones_obra c
           ON c.titulo_origen = p.titulo
          AND c.autor_origen  = p.autor
""")

## Tabla dimensión obra

In [73]:
con.sql("""
    CREATE OR REPLACE TABLE dim_obra AS
    WITH obra AS (
        SELECT
            id_obra,
            titulo,
            autor,
            autor = '[sin autor]' AS sin_autor,
            CAST(MIN(fecha_prestamo) AS DATE)  AS fecha_primer_prestamo,
            MIN(anio_obra) AS anio_obra,
            COUNT(*)  AS n_prestamos
        FROM prestamos_obra
        GROUP BY id_obra, titulo, autor
    ),
    clasif AS (
        SELECT *,
            CAST(date_trunc('month', fecha_primer_prestamo) AS DATE) AS mes_primer_prestamo,
            year(fecha_primer_prestamo)  AS anio_primer_prestamo,
            year(fecha_primer_prestamo) * 12
              + month(fecha_primer_prestamo) AS mes_indice_primer,
        FROM obra
    )
    SELECT
        c.id_obra,
        c.titulo,
        c.autor,
        c.fecha_primer_prestamo,
        c.mes_primer_prestamo,
        c.anio_primer_prestamo,
        c.mes_indice_primer,
        c.anio_obra,
        e.tipo_corregido AS tipo_corregido_manual,
        CAST(c.n_prestamos AS BIGINT) AS n_prestamos_total
    FROM clasif c
    LEFT JOIN excepciones_tipo_obra e
           ON e.titulo = c.titulo
          AND e.autor  = c.autor
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [75]:
con.sql("""
    SELECT
      COUNT(*) AS n_obras,
      COUNT(DISTINCT id_obra) AS ids_unicos,
      SUM(n_prestamos_total) AS total_prestamos
    FROM dim_obra
""").show()

┌─────────┬────────────┬─────────────────┐
│ n_obras │ ids_unicos │ total_prestamos │
│  int64  │   int64    │     int128      │
├─────────┼────────────┼─────────────────┤
│  301313 │     301313 │        16336911 │
└─────────┴────────────┴─────────────────┘



## Tabla prestamos agregagos por obra y año

In [76]:
con.sql("""
    CREATE OR REPLACE TABLE agg_obra_anio AS
    SELECT
        p.id_obra,
        year(p.fecha_prestamo) AS anio,
        CAST(COUNT(*) AS BIGINT) AS n_prestamos,
        CAST(COUNT(*) FILTER (WHERE p.carnet_adulto = '0') AS BIGINT) AS n_prestamos_infantil,
        CAST(COUNT(DISTINCT p.id_ejemplar) AS BIGINT) AS n_ejemplares_prestados
    FROM prestamos_obra p
    JOIN dim_obra d USING (id_obra)
    GROUP BY ALL
""")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

### Lista fusiones

In [78]:
# Lista fusiones primera revision:
con.sql("""
    CREATE OR REPLACE TABLE lista_fusiones AS
    SELECT md5(titulo || ' | ' || autor) AS id_obra, titulo, autor, tipo_fusion,
           'revision_08_11' AS origen
    FROM revision_top_anterior
    WHERE tipo_fusion IS NOT NULL
""")

# 1) Lista fusiones encontradas segunda revision:
con.sql("""
    INSERT INTO lista_fusiones
    SELECT d.id_obra, d.titulo, d.autor, n.tipo_fusion, 'revision_11_11'
    FROM (VALUES
        ('chainsaw man',                                   'fujimoto, tatsuki', 'serie'),
        ('haikyu',                                         'furudate, haruichi','serie'),
        ('guardianes de la noche kimetsu no yaiba',        'gotoge, koyoharu',  'serie'),
        ('mafalda tiras de quino',                         'quino',             'serie'),
        ('mundial',                                        'ibanez, f',         'serie'),
        ('pokemon rojo verde y azul',                      'kusaka, hidenori',  'serie'),
        ('spy x family',                                   'endo, tatsuya',     'serie'),
        ('death note',                                     'oba, tsugumi',      'serie'),
        ('keroro',                                         'yoshizaki, mine',   'serie'),
        ('yo kai watch',                                   'konishi, noriyuki', 'serie'),
        ('el capitan calzoncillos y la gran batalla contra el mocoso c', 'pilkey, dav', 'serie'),
        ('grecia',                                         '[sin autor]',       'guía'),
        ('londres',                                        '[sin autor]',       'guía')
    ) AS n(titulo, autor, tipo_fusion)
    JOIN dim_obra d ON d.titulo = n.titulo AND d.autor = n.autor
""")

#TOP50

In [83]:
con.sql(f"""
    CREATE OR REPLACE TABLE titulo_mostrar_1111 AS
    SELECT *
    FROM read_csv('{RUTA_SILVER/'verificacion/titulo_mostrar_1111.csv'}',
                  delim   = ',',
                  header  = true,
                  quote   = '"',          -- los autores ("kinney, jeff") van entre comillas
                  columns = {{'titulo': 'VARCHAR', 'autor': 'VARCHAR',
                              'titulo_mostrar': 'VARCHAR', 'revisar': 'VARCHAR'}})
""")

In [84]:
con.sql("""
    CREATE OR REPLACE TABLE ranking_sin_fusiones AS
    SELECT
    a.anio,
    a.id_obra,
    a.n_prestamos,
    RANK() OVER (PARTITION BY a.anio ORDER BY a.n_prestamos DESC) AS posicion
    FROM agg_obra_anio a
    WHERE a.id_obra NOT IN (SELECT id_obra FROM lista_fusiones)
    QUALIFY posicion <= 60
""")

In [85]:
con.sql("""
    CREATE OR REPLACE TABLE top_anual_volumen AS
    SELECT anio, id_obra, n_prestamos,
           RANK() OVER (PARTITION BY anio ORDER BY n_prestamos DESC) AS posicion
    FROM agg_obra_anio
    WHERE id_obra NOT IN (SELECT id_obra FROM lista_fusiones)
    QUALIFY posicion <= 50
""")

In [87]:
# dim_obra_top: una fila por obra del top con título para mostrar
con.sql("""
    CREATE OR REPLACE TABLE dim_obra_top AS
    WITH tm AS (
        SELECT titulo, autor,
        titulo_mostrar
        FROM revision_top_anterior
        WHERE tipo_fusion IS NULL
        UNION ALL
        SELECT titulo, autor, titulo_mostrar
        FROM titulo_mostrar_1111
    )
    SELECT
        d.id_obra,
        COALESCE(tm.titulo_mostrar, d.titulo)  AS titulo_mostrar,
        d.autor,
        d.anio_obra,
        (tm.titulo_mostrar IS NULL)            AS sin_titulo_mostrar
    FROM dim_obra d
    LEFT JOIN tm ON tm.titulo = d.titulo AND tm.autor = d.autor
    WHERE d.id_obra IN (SELECT id_obra FROM top_anual_volumen)
""")

# top_obra_anio: obras del top 50 por año con ejemplares, préstamos por ejemplar y peso de cada carné
con.sql("""
    CREATE OR REPLACE TABLE top_obra_anio AS
    SELECT
        t.anio,
        t.id_obra,
        o.titulo_mostrar,
        o.autor,
        t.posicion,
        t.n_prestamos,
        a.n_ejemplares_prestados,
        t.n_prestamos::DOUBLE / a.n_ejemplares_prestados       AS prestamos_por_ejemplar,
        100.0 * a.n_prestamos_infantil / a.n_prestamos         AS pct_infantil,
        100.0 - 100.0 * a.n_prestamos_infantil / a.n_prestamos AS pct_adulto
    FROM top_anual_volumen t
    JOIN agg_obra_anio a USING (id_obra, anio)
    JOIN dim_obra_top  o USING (id_obra)
    ORDER BY t.anio, t.posicion
""")

## Obras 10%

In [88]:
# Prestamos, obras y ejemplares en 10% obras mas prestadas. Sin series nie guias
con.sql("""WITH base AS (
    SELECT
      anio,
      id_obra,
      n_prestamos
    FROM agg_obra_anio
    WHERE anio BETWEEN 2018 AND 2025
      AND id_obra NOT IN (SELECT id_obra FROM lista_fusiones)
),
umbral AS (
    SELECT
      anio,
      QUANTILE_DISC(n_prestamos, 0.9) AS umbral_p90
    FROM base
    GROUP BY anio
),
por_anio AS (
    SELECT
      b.anio,
      COUNT(*) AS n_obras,
      SUM(b.n_prestamos) AS n_prestamos,
      ROUND(AVG(b.n_prestamos), 2) AS media_prestamos_obra,
      MEDIAN(b.n_prestamos) AS mediana_prestamos_obra,
      u.umbral_p90,
      COUNT(*) FILTER (WHERE b.n_prestamos >= u.umbral_p90) AS n_obras_p90,
      SUM(b.n_prestamos) FILTER (WHERE b.n_prestamos >= u.umbral_p90) AS n_prestamos_p90
    FROM base b
    JOIN umbral u USING (anio)
    GROUP BY b.anio, u.umbral_p90
)
SELECT *,
       ROUND(100.0 * n_obras_p90     / n_obras,     1) AS pct_obras_p90,
       ROUND(100.0 * n_prestamos_p90 / n_prestamos, 1) AS pct_prestamos_p90
FROM por_anio
ORDER BY anio;""")

┌───────┬─────────┬─────────────┬──────────────────────┬────────────────────────┬────────────┬─────────────┬─────────────────┬───────────────┬───────────────────┐
│ anio  │ n_obras │ n_prestamos │ media_prestamos_obra │ mediana_prestamos_obra │ umbral_p90 │ n_obras_p90 │ n_prestamos_p90 │ pct_obras_p90 │ pct_prestamos_p90 │
│ int64 │  int64  │   int128    │        double        │         double         │   int64    │    int64    │     int128      │    double     │      double       │
├───────┼─────────┼─────────────┼──────────────────────┼────────────────────────┼────────────┼─────────────┼─────────────────┼───────────────┼───────────────────┤
│  2018 │  170603 │     2017501 │                11.83 │                    3.0 │         26 │       17085 │         1266982 │          10.0 │              62.8 │
│  2019 │  168732 │     2190031 │                12.98 │                    3.0 │         28 │       17213 │         1397953 │          10.2 │              63.8 │
│  2020 │  131547 │   

In [89]:
con.sql("""SELECT ROUND(AVG(n_obras), 0) AS media_obras_anio,
       MEDIAN(n_obras)        AS mediana_obras_anio
FROM (
    SELECT anio, COUNT(*) AS n_obras
    FROM agg_obra_anio
    WHERE anio BETWEEN 2018 AND 2025
    GROUP BY anio
) """)

┌──────────────────┬────────────────────┐
│ media_obras_anio │ mediana_obras_anio │
│      double      │       double       │
├──────────────────┼────────────────────┤
│         159367.0 │           165932.5 │
└──────────────────┴────────────────────┘

In [ ]:
RUTA_GOLD     = '/content/drive/MyDrive/0_PROYECTO_FINAL/ESTRUCTURA_MEDALLION/3_GOLD/FINAL'
RUTA_PARQUET  = f'{RUTA_GOLD}/parquet'
for r in (RUTA_PARQUET):
    os.makedirs(r, exist_ok=True)

parquet_y_csv = ['dim_calendario', 'dim_biblioteca', 'dim_distrito_anio', 'dim_obra_top',
                 'prestamos_mes_ag', 'prestamos_mes_total', 'h1_indice_anual',
                 'estacionalidad_ratios', 'carnet_indice_anual',
                 'top_anual_volumen', 'top_obra_anio']


In [91]:
def exportar(tabla, carpeta, formato):
    """Exporta y relee para comprobar que el número de filas coincide."""
    if formato == 'parquet':
        ruta = f'{carpeta}/{tabla}.parquet'
        con.sql(f"COPY (SELECT * FROM {tabla}) TO '{ruta}' (FORMAT PARQUET)")
        n_fich = con.sql(f"SELECT COUNT(*) FROM read_parquet('{ruta}')").fetchone()[0]
    else:
        ruta = f'{carpeta}/{tabla}.csv'
        con.sql(f"COPY (SELECT * FROM {tabla}) TO '{ruta}' (HEADER, DELIMITER ';')")
        n_fich = con.sql(f"SELECT COUNT(*) FROM read_csv('{ruta}', delim=';', header=true, all_varchar=true)").fetchone()[0]
    n_tabla = con.sql(f"SELECT COUNT(*) FROM {tabla}").fetchone()[0]
    print(f"{'OK ' if n_fich == n_tabla else 'ERROR'} {formato:7} {tabla:32} {n_tabla:>10,} filas")

for t in parquet_y_csv:
    exportar(t, RUTA_PARQUET, 'parquet')

OK  parquet dim_calendario                           96 filas
OK  parquet dim_biblioteca                           55 filas
OK  parquet dim_distrito_anio                       168 filas
OK  parquet dim_obra_top                            152 filas
OK  parquet prestamos_mes_ag                      8,711 filas
OK  parquet prestamos_mes_total                      96 filas
OK  parquet h1_indice_anual                           8 filas
OK  parquet estacionalidad_ratios                    94 filas
OK  parquet carnet_indice_anual                       8 filas
OK  parquet top_anual_volumen                       403 filas
OK  parquet top_obra_anio                           403 filas
